# 02 — dist_autoへのGPR適用

元コードと同じく、1つのtagを丸ごとテストにして、残りtagから予測します。既定は `TEST_TAG='10'` です。予測平均に加え、不確実性と95%区間を作ります。

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO_URL = 'https://github.com/futoshi-futami/Chemistory.git'
REPO_REF = os.environ.get('CHEMISTORY_REF', 'main')
FALLBACK_REF = 'agent/rbf-kernel-comparison'  # PR #2。mainへmerge後はfallback不要
cwd = Path.cwd()
if (cwd / 'pyproject.toml').exists():
    PROJECT_ROOT = cwd
elif (Path('/content/Chemistory') / 'pyproject.toml').exists():
    PROJECT_ROOT = Path('/content/Chemistory')
elif 'google.colab' in sys.modules:
    PROJECT_ROOT = Path('/content/Chemistory')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    raise FileNotFoundError('Chemistory project rootでノートブックを実行してください。')
# PRのレビュー中も実行可能にし、merge後は自動的にmainを使います。
if 'google.colab' in sys.modules and not (PROJECT_ROOT / 'src/chemistory_gpr/kernels.py').exists():
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'fetch', '--depth', '1', 'origin', FALLBACK_REF], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable, 'scripts/prepare_data.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT_ROOT)], check=True)
src_dir = str(PROJECT_ROOT / 'src')
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import importlib
importlib.invalidate_caches()
import chemistory_gpr
print('PROJECT_ROOT =', PROJECT_ROOT)
print('chemistory_gpr =', chemistory_gpr.__file__)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from IPython.display import display
from chemistory_gpr.dist_auto import (
    default_dist_auto_candidates, fit_held_out_tag, leave_one_tag_out,
    load_dist_auto_data, make_grid, predict_grid, standardized_tag_centroid_distances,
    summarize_dist_auto_metrics,
)

DATA_DIR = PROJECT_ROOT / 'data' / 'dist_auto'
RESULTS = PROJECT_ROOT / 'results'
RESULTS.mkdir(exist_ok=True)
TEST_TAG = '10'
GRID_SIZE = 30
ARD_RESTARTS = 0  # 元コードどおりは5。0でも第二種最尤法による最適化は実施
RUN_FULL_LOTO = False  # Trueで全tag×全kernelを再計算（Colabでは時間を要します）

## A. データとtag外挿設定

In [ ]:
data = load_dist_auto_data(DATA_DIR)
summary = pd.DataFrame({'tag': data.tags, 'y': data.y}).groupby('tag', sort=False)['y'].agg(['count','mean','std','min','max'])
print('common Xmat features =', len(data.feature_columns))
display(summary)
centroid_distances = standardized_tag_centroid_distances(data)
display(centroid_distances.style.format('{:.3f}'))

`tag=b` は目的変数の平均・分散に加え、標準化Xmat空間の重心も他tagから大きく離れます。これは物理条件名を特定するものではありませんが、構造領域外挿であることを支持します。したがって、tag 10だけでなく全tagのleave-one-tag-out診断も後で確認します。

## B. 指定tagを完全にhold outしてカーネルを比較

前回モデルに加え、Xmatだけを訓練foldで標準化する元の前処理に揃えて、Matérn 1/2、Matérn 3/2、等方RBF、RBF-ARDを比較します。

In [ ]:
candidates = default_dist_auto_candidates(rbf_ard_restarts=ARD_RESTARTS)
test_rows, candidate_models, candidate_heldout = [], {}, {}
for config in candidates:
    print('Running', config.name)
    fitted, prediction, metrics = fit_held_out_tag(data, TEST_TAG, config)
    candidate_models[config.name] = fitted
    candidate_heldout[config.name] = prediction
    test_rows.append(metrics)
comparison = pd.DataFrame(test_rows).sort_values('R2', ascending=False)
comparison.to_csv(RESULTS / f'dist_auto_test_{TEST_TAG}_kernel_comparison.csv', index=False)
display(comparison[['model','kernel_family','ard','optimizer_restarts','R2','corr2','RMSE','MAE','coverage_95','length_scales_at_upper_bound','length_scale_count']])
best_name = comparison.iloc[0]['model']
best_config = next(c for c in candidates if c.name == best_name)
model = candidate_models[best_name]
heldout = candidate_heldout[best_name]
heldout.to_csv(RESULTS / f'dist_auto_test_{TEST_TAG}_predictions.csv', index=False)
print('Selected for the surface:', best_name)
print('optimized kernel:', comparison.iloc[0]['optimized_kernel'])

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.errorbar(heldout['y'], heldout['pred_mean'], yerr=1.96*heldout['pred_std'], fmt='o', ms=4, alpha=.7)
lims = [min(heldout['y'].min(), heldout['lower_95'].min()), max(heldout['y'].max(), heldout['upper_95'].max())]
ax.plot(lims, lims, '--', color='black')
ax.set(xlabel='Observed y', ylabel='Predicted y', title=f'Held-out tag {TEST_TAG}: mean ± 95%')
plt.show()

この比較ではhold-out tagの真値を見てカーネルを選んでいます。したがって最良値は候補比較値であり、選択後の独立テスト値ではありません。次の新規tagまたは構造では、選んだカーネルを事前固定して評価してください。

## C. leave-one-tag-out診断

全tag比較は高次元RBF-ARDのため時間がかかります。`RUN_FULL_LOTO=True` なら再計算し、FalseならGitHubに保存済みの結果表を読みます。

In [ ]:
if RUN_FULL_LOTO:
    all_predictions, all_metrics = [], []
    for config in candidates:
        print('Running all held-out tags:', config.name)
        group_oof, group_metrics = leave_one_tag_out(data, config, n_jobs=1)
        group_oof.insert(0, 'model', config.name)
        all_predictions.append(group_oof)
        all_metrics.append(group_metrics)
    pd.concat(all_predictions, ignore_index=True).to_csv(RESULTS / 'dist_auto_kernel_comparison_predictions.csv', index=False)
    group_metrics = pd.concat(all_metrics, ignore_index=True)
    group_metrics.to_csv(RESULTS / 'dist_auto_kernel_comparison_metrics.csv', index=False)
else:
    group_metrics = pd.read_csv(RESULTS / 'dist_auto_kernel_comparison_metrics.csv')
display(group_metrics[['model','test_tag','kernel_family','ard','R2','corr2','RMSE','MAE','coverage_95']])
group_summary = summarize_dist_auto_metrics(group_metrics)
group_summary.to_csv(RESULTS / 'dist_auto_kernel_comparison_summary.csv', index=False)
display(group_summary)

`corr2` は元コードとの比較用です。主指標は `R2=1-SSE/SST` です。相関が高くても平均がずれると `corr2` は高いまま `R2` が悪化するので、両者を混同しないでください。

保存済み結果ではRBF-ARDがtag 10–25の4条件でR²首位、等方RBFがtag別平均R²と全OOF R²で僅差の首位、Matérn 3/2が95% coverageで首位です。tag bの最良R²は負のままで、カーネル変更だけでは解決していません。

## D. 新しいxyグリッドの予測面と不確実性

In [ ]:
grid, grid_features = make_grid(DATA_DIR, TEST_TAG, data.feature_columns, grid_size=GRID_SIZE)
surface = predict_grid(model, grid, grid_features)
surface.insert(0, 'model', best_name)
surface.to_csv(RESULTS / f'dist_auto_surface_{TEST_TAG}.csv', index=False)
mean_pivot = surface.pivot(index='y', columns='x', values='pred_mean')
std_pivot = surface.pivot(index='y', columns='x', values='pred_std')
fig = go.Figure(go.Surface(
    x=mean_pivot.columns, y=mean_pivot.index, z=mean_pivot.to_numpy(),
    surfacecolor=std_pivot.to_numpy(), colorbar={'title':'predictive std'},
))
fig.update_layout(title=f'GPR mean surface — held-out tag {TEST_TAG}', scene={'xaxis_title':'x','yaxis_title':'y','zaxis_title':'prediction'})
fig.show()

In [ ]:
fig = go.Figure(go.Heatmap(x=std_pivot.columns, y=std_pivot.index, z=std_pivot.to_numpy(), colorbar={'title':'std'}))
fig.update_layout(title='Predictive uncertainty', xaxis_title='x', yaxis_title='y')
fig.show()

best_mean = surface.loc[surface['pred_mean'].idxmax(), ['x','y','pred_mean','pred_std','lower_95']]
best_lcb = surface.loc[surface['lower_confidence_bound'].idxmax(), ['x','y','pred_mean','pred_std','lower_confidence_bound']]
print('Maximum predictive mean:\n', best_mean.to_string())
print('\nMaximum 95% lower confidence bound:\n', best_lcb.to_string())

### 回転角について

このColab版は検証可能な `angle=0` を対象にします。受領ZIPの `rotate_xyz.exe` はWindows専用で、同梱の `source.xyz` と `*-altered.xyz` はNULのみでした。非ゼロ回転を厳密に追加するには、回転プログラムのCソースまたは正常な回転前後XYZが必要です。